<a href="https://colab.research.google.com/github/ChiaHeJun/code/blob/main/WalkthroughTask4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walkthrough: Task 4 - JP House Allocation System

SQLite plus a Flask web app. Run this notebook **from the Suggested Solution folder**, top to bottom. Each section follows the same shape: what the question is really asking, the thinking that leads to the design, then the solution built up and run. The goal is not to memorise the code but to be able to re-derive it.

---
## Task 4.1 - design and create the database

**What is asked.** Create `HOUSEALLOCATION.db` with a HOUSE table and a STAFF table.

**Thinking like a database designer.** Two kinds of thing exist here: houses and staff. Each becomes a table. Then three key decisions:

- What identifies a house? Its name; no two houses share one. So `houseName` is HOUSE's PRIMARY KEY.
- What identifies a staff member? Names can repeat, so the school gave everyone a `staffID`: that is STAFF's PRIMARY KEY.
- How do the tables relate? A staff member belongs to (at most) one house, so STAFF carries a `houseName` column, and declaring it a FOREIGN KEY referencing `HOUSE(houseName)` tells the database that value must be a real house. The FK lives on the **many** side (STAFF), pointing at the **one** side (HOUSE).

One practical line: `DROP TABLE IF EXISTS` first, so the cell can be rerun during development without a "table already exists" error. `IF EXISTS` matters; a bare `DROP TABLE` fails on the very first run when there is nothing to drop. And the filename must be exactly `HOUSEALLOCATION.db`: the marking, and the Flask app later, both look for that name.

In [ ]:
#Task 4.1
import sqlite3

conn = sqlite3.connect("HOUSEALLOCATION.db")

conn.execute("DROP TABLE IF EXISTS HOUSE")
conn.execute("DROP TABLE IF EXISTS STAFF")

conn.execute('''CREATE TABLE HOUSE(
houseName TEXT PRIMARY KEY,
imageURL TEXT)''')

conn.execute('''CREATE TABLE STAFF(
staffID TEXT PRIMARY KEY,
name TEXT,
department TEXT,
houseName TEXT,
FOREIGN KEY (houseName) REFERENCES HOUSE(houseName))''')

conn.commit()
conn.close()

In [ ]:
# Verify the schema actually says what we intended
conn = sqlite3.connect("HOUSEALLOCATION.db")
for table in ("HOUSE", "STAFF"):
    print(table, "columns:", [ (r[1], "PK" if r[5] else "") for r in conn.execute("PRAGMA table_info(%s)" % table) ])
print("STAFF foreign keys:", list(conn.execute("PRAGMA foreign_key_list(STAFF)")))
conn.close()

---
## Task 4.2 - load the text files

**What is asked.** Populate both tables from `HOUSE.txt` and `STAFF.txt`.

**Thinking.** The reading pattern is Task 1.1 again: line by line, `strip().split(',')`. The inserting pattern is the one professional habit this paper insists on: **parameterised SQL**. `INSERT INTO HOUSE VALUES (?,?)` with the values passed as a tuple lets the database library handle quoting, so a name containing an apostrophe cannot break the statement (and hostile input cannot rewrite it). Building the SQL by string concatenation loses the mark. A convenient detail: `split` already produces a list of four fields in column order, so the STAFF insert can pass `line` directly. One `commit()` after the loops makes the inserts permanent.

In [ ]:
# task 4.2
conn = sqlite3.connect("HOUSEALLOCATION.db")

file = open("HOUSE.txt",'r')
for line in file:
    line = line.strip().split(',')
    conn.execute("INSERT INTO HOUSE VALUES (?,?)",(line[0],line[1]))
file.close()
file = open("STAFF.txt",'r')
for line in file:
    line = line.strip().split(',')

    conn.execute("INSERT INTO STAFF VALUES (?,?,?,?)",line)

conn.commit()
file.close()
conn.close()

In [ ]:
# Verify: 6 houses, 10 staff
conn = sqlite3.connect("HOUSEALLOCATION.db")
print("HOUSE rows:", conn.execute("SELECT COUNT(*) FROM HOUSE").fetchone()[0])
print("STAFF rows:", conn.execute("SELECT COUNT(*) FROM STAFF").fetchone()[0])
conn.close()

---
## Task 4.3 - department lookup with aligned output

**What is asked.** Ask for a department, list its staff with staff ID, name and house, in neat columns with headings.

**Thinking.** The query filters with `WHERE department=?`, parameterised as before, and `fetchall()` collects the matching rows. The display problem is alignment: `print(a, b)` separates fields by one space, so columns drift with name length. `ljust(n)` pads a string to a fixed width, and using the same widths for the heading row and every data row is the whole trick. In the exam you type the department; the run below scripts `MATH/COMP`, the department the question tests.

In [ ]:
# Task 4.3
from unittest import mock

def task4_3():
    conn = sqlite3.connect("HOUSEALLOCATION.db")
    dept = input("Enter dept: ")
    cursor = conn.execute("SELECT * FROM STAFF WHERE department=?", (dept,))
    data = cursor.fetchall()
    print("Staff ID".ljust(9), "Staff Name".ljust(15), "House Name")
    for line in data:
            print(line[0].ljust(9), line[1].ljust(15), line[3])

    conn.close()

print("Simulating input MATH/COMP:")
with mock.patch("builtins.input", side_effect=["MATH/COMP"]):
    task4_3()

---
## Task 4.4 - the Flask web app

**What is asked.** A menu page linking to three features: display house logos, allocate a house, display teachers by house.

**Think in routes and templates before writing anything.** Each page is a pair: a route function (Python: fetch data, decide) and a template (HTML: display). Map the whole app first:

| URL | Route function | Template | Job |
|---|---|---|---|
| `/` | `index` | `index.html` | menu of three links |
| `/displayLogo` | `displayLogo` | `displayLogo.html` | table of houses and logos |
| `/allocate_form1` | `allocate_form1` | `allocate1.html` | the name/department form |
| `/allocateResult` | `allocate` (POST) | replies directly | do the allocation |
| `/displayTeachersByHouse` | `display` | `teachers.html` | six-column table |

Two syntax details killed several cohort submissions before a single page rendered: `@app.route(...)` is a **decorator**, written with `@` on the line above the function, and `if __name__ == "__main__":` needs `==`. A missing `templates` folder is the third instant killer: `render_template("index.html")` looks only inside a folder literally named `templates` next to the .py file.

The cell below is the full app from `Task4_4.py` with one difference: **no `app.run()`**, because a notebook cell running a server never returns. Defining the app is harmless, and Flask's built-in test client can then request pages right here, which is how each route is checked below without leaving the notebook.

In [ ]:
import sqlite3, os
from flask import Flask, render_template, request
import random

app = Flask(__name__)
app.root_path = os.getcwd()   # notebook only: make Flask look for templates/ here

@app.route('/')
def index():
    return render_template("index.html")

@app.route('/displayLogo')
def displayLogo():
    conn = sqlite3.connect("HOUSEALLOCATION.db")

    # retrieve house names and images
    cursor = conn.execute("SELECT * FROM HOUSE")
    houses = cursor.fetchall()

    conn.close()
    return render_template("displayLogo.html", houses=houses)

@app.route('/allocate_form1')
def allocate_form1():
    return render_template("allocate1.html")

@app.route("/allocateResult", methods=["POST"])
def allocate():
    name = request.form["name"]
    dept = request.form["dept"]

    conn = sqlite3.connect("HOUSEALLOCATION.db")
    cursor = conn.execute('''SELECT * FROM STAFF
                 WHERE name = ?
                 AND department = ?
                 ''', (name, dept))
    staff = cursor.fetchone()

    if staff[3] != "Unassigned":  # staff already belongs to a House
        housename = staff[3]
        conn.close()
        return "Teacher " + name + " from " + dept + " is already in " + housename + " house."

    else:  # house is unassigned, randomly assign one
        cursor = conn.execute("SELECT * FROM HOUSE")
        houses = cursor.fetchall()
        num = random.randint(0, 5)

        housename = houses[num][0]
        staffID = staff[0]
        conn.execute('''UPDATE STAFF SET houseName = ? WHERE staffID = ?''', (housename, staffID))
        conn.commit()
        conn.close()

        return "Teacher " + name + " from " + dept + " is randomly assigned to " + housename + " house."

@app.route('/displayTeachersByHouse')
def display():
    houseAndTeachers = []
    conn = sqlite3.connect("HOUSEALLOCATION.db")
    cursor = conn.execute("SELECT houseName FROM HOUSE")
    houses = cursor.fetchall()
    for house in houses:
        cursor = conn.execute("SELECT name FROM STAFF WHERE houseName = ?", (house[0],))
        teachers = cursor.fetchall()
        houseAndTeachers.append([house[0], teachers])
    conn.close()
    return render_template("teachers.html", houseAndTeachers=houseAndTeachers)

print("App defined. In Task4_4.py the file ends with app.run(), which starts the server.")

**How each route thinks.**

**The menu (`/`).** Three anchor tags whose `href` values are the other routes' URLs. The link text between `<a>` and `</a>` is what the user clicks; leave it empty and the menu looks blank.

**The logos page.** The route's job is data: `SELECT * FROM HOUSE`, `fetchall()`, pass the rows in as `houses=houses`. The template's job is display: a header row `<th>House</th><th>Logo</th>`, then a Jinja loop, one table row per house:

```html
{% for house in houses %}
    <tr>
    <td>{{house[0]}}</td>
    <td><img src="{{house[1]}}" width="100" height="70"></td>
    </tr>
{% endfor %}
```

`{% %}` is control flow, `{{ }}` prints a value. Each `house` is one row tuple, so `house[0]` is the name and `house[1]` the image URL, dropped straight into `src` with the required `width="100" height="70"`. Hardcoding six `<img>` tags instead of looping over the database earned the layout marks but lost the retrieval mark.

**The form.** `allocate1.html` has two inputs whose **`name` attributes** (`name`, `dept`) are the keys the POST route reads from `request.form`, and its `action="/allocateResult"` must be a URL that exists. A form posting to a nonexistent endpoint, or reading keys that do not match the input names, fails at the first click; both happened in the cohort.

**The allocation route.** Look the teacher up with a two-condition parameterised query. Then one decision: `staff[3]` (the houseName column) not `"Unassigned"` means already allocated, so reply with the "is already in" sentence. Otherwise pick a random house, `UPDATE ... SET houseName = ? WHERE staffID = ?` (updating by primary key, so exactly one row changes), `commit()`, and reply with the "is randomly assigned to" sentence. Both sentences are marked word for word. A cohort favourite: swapping the two branches, so an unassigned teacher is told they already have a house.

**Teachers by house.** The page needs, for each house, the teachers in it: a list of `[houseName, [teachers]]` pairs, built with one query per house. The template then loops **twice** over the same list: once to emit the six `<th>` headers, once to emit six `<td>` cells, each with an inner loop printing that house's teachers. Unassigned staff match no house and appear nowhere, which is what the question wants.

Now request every page through the test client, without a browser:

In [ ]:
client = app.test_client()

print("GET /  ->", client.get("/").status)
print(client.get("/").get_data(as_text=True))

In [ ]:
r = client.get("/displayLogo")
print("GET /displayLogo ->", r.status)
print(r.get_data(as_text=True)[:400])

In [ ]:
r = client.get("/allocate_form1")
print("GET /allocate_form1 ->", r.status)
print(r.get_data(as_text=True))

In [ ]:
# The two allocation outcomes. Sandra Chelvan starts Unassigned; reset her
# afterwards so this cell can be rerun.
r = client.post("/allocateResult", data={"name": "Sandra Chelvan", "dept": "BIO/PHY"})
print(r.get_data(as_text=True))

r = client.post("/allocateResult", data={"name": "Low Jia Ying", "dept": "MATH/COMP"})
print(r.get_data(as_text=True))

conn = sqlite3.connect("HOUSEALLOCATION.db")
conn.execute("UPDATE STAFF SET houseName = 'Unassigned' WHERE staffID = 'S001'")
conn.commit()
conn.close()

In [ ]:
r = client.get("/displayTeachersByHouse")
print("GET /displayTeachersByHouse ->", r.status)
print(r.get_data(as_text=True))

---
## Task 4.5 - run the app and save the output

Run the real thing from a terminal in this folder: `python Task4_4.py`, then open http://127.0.0.1:5000 in a browser. Click through the three features, performing the allocations the question specifies, and **save each results page from the browser** (Ctrl+S). Those saved pages are the evidence the app ran; hand-written HTML files are not, and this final mark was the single most commonly dropped mark in the cohort.

---
## The thread through Task 4

The database is designed once and then everything reads or writes it through the same two habits: parameterised SQL and exact names (the db filename, template filenames, form field names, endpoint names). Nearly every Flask failure in the cohort was a name that did not match its counterpart. When a page breaks, diff the names first.